# Step 7: Round 1c Allowed Aggregate Summary

这个 notebook 不做模型推理，只读取已经冻结的 `Round 1b` 结果和 `Round 1c` 规则，自动生成 allowed aggregate views 下的摘要。


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path


## 1. 配置路径


In [ ]:
PROJECT_ROOT_OVERRIDE = ''


def detect_project_root():
    candidates = []
    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/kaggle/working/2026_SelectTransfer'),
    ])
    seen = set()
    checked = []
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(str(candidate))
        if (candidate / 'results' / '05_round1b_prep' / 'round1c_role_aware_smoke_table.csv').exists() and (candidate / 'results' / '05_round1b_prep' / 'round1c_aggregate_rules.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root. Checked: ' + ' | '.join(checked))

PROJECT_ROOT = detect_project_root()
INPUT_DIR = PROJECT_ROOT / 'results' / '05_round1b_prep'
OUTPUT_DIR = PROJECT_ROOT / 'results' / '06_round1c_summary'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROLE_TABLE_PATH = INPUT_DIR / 'round1c_role_aware_smoke_table.csv'
SUBSET_PATH = INPUT_DIR / 'round1c_role_aware_smoke_subset.csv'
RULES_PATH = INPUT_DIR / 'round1c_aggregate_rules.md'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 2. 读取 role-aware 表与 subset


In [ ]:
def read_csv(path):
    with path.open() as f:
        return list(csv.DictReader(f))

role_rows = read_csv(ROLE_TABLE_PATH)
subset_rows = read_csv(SUBSET_PATH)
subset_by_task = {r['task_id']: r for r in subset_rows}
print('role rows =', len(role_rows))
print('subset tasks =', len(subset_rows))


## 3. 按 allowed aggregate views 生成摘要表


In [ ]:
process_rows = []
diagnostic_rows = []
audit_rows = []
overview_rows = []

by_task = defaultdict(list)
for row in role_rows:
    by_task[row['target_task_id']].append(row)

for task_id, rows in by_task.items():
    meta = subset_by_task[task_id]
    bucket = meta['subset_bucket']
    total_runs = len(rows)
    memory_runs = sum(1 for r in rows if r['memory_attached'] == 'true')
    memory_verbalized = sum(1 for r in rows if r['memory_verbalized'] == 'yes')
    explicit_use = sum(1 for r in rows if r['explicit_use'] == 'yes')
    explicit_reject = sum(1 for r in rows if r['explicit_reject'] == 'yes')
    outcome_changed = sum(1 for r in rows if r['outcome_changed_vs_baseline'] == 'yes')
    improved = sum(1 for r in rows if r['improved_vs_baseline'] == 'yes')
    degraded = sum(1 for r in rows if r['degraded_vs_baseline'] == 'yes')
    answer_changed = sum(1 for r in rows if r['answer_changed_vs_baseline'] == 'yes')
    base = {
        'task_id': task_id,
        'target_cluster': rows[0]['target_cluster'],
        'subset_bucket': bucket,
        'subset_priority': meta['subset_priority'],
        'run_in_round1c': meta['run_in_round1c'],
        'report_in_aggregate': meta['report_in_aggregate'],
        'report_as': meta['report_as'],
        'total_runs': total_runs,
        'memory_runs': memory_runs,
        'memory_verbalized_runs': memory_verbalized,
        'explicit_use_runs': explicit_use,
        'explicit_reject_runs': explicit_reject,
        'outcome_changed_runs': outcome_changed,
        'improved_runs': improved,
        'degraded_runs': degraded,
        'answer_changed_runs': answer_changed,
        'notes': meta['notes'],
    }
    if bucket == 'process_sanity':
        process_rows.append(base)
    elif bucket in {'artifact_sensitive_diagnosis', 'answer_format_diagnosis'}:
        diagnostic_rows.append(base)
    elif bucket == 'audit_boundary':
        audit_rows.append(base)

for bucket_name, rows in [('process_sanity', process_rows), ('diagnostic', diagnostic_rows), ('audit_boundary', audit_rows)]:
    if not rows:
        continue
    overview_rows.append({
        'subset_bucket': bucket_name,
        'task_count': len(rows),
        'run_count': sum(int(r['total_runs']) for r in rows),
        'memory_runs': sum(int(r['memory_runs']) for r in rows),
        'memory_verbalized_runs': sum(int(r['memory_verbalized_runs']) for r in rows),
        'explicit_use_runs': sum(int(r['explicit_use_runs']) for r in rows),
        'explicit_reject_runs': sum(int(r['explicit_reject_runs']) for r in rows),
        'outcome_changed_runs': sum(int(r['outcome_changed_runs']) for r in rows),
        'improved_runs': sum(int(r['improved_runs']) for r in rows),
        'degraded_runs': sum(int(r['degraded_runs']) for r in rows),
    })

print('process rows =', len(process_rows))
print('diagnostic rows =', len(diagnostic_rows))
print('audit rows =', len(audit_rows))


## 4. 写出 csv 与 markdown 摘要


In [ ]:
def write_csv(path, rows):
    if not rows:
        return
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

def md_table(rows, cols):
    if not rows:
        return '_No rows._\n'
    header = '| ' + ' | '.join(cols) + ' |\n'
    sep = '|' + '|'.join(['---'] * len(cols)) + '|\n'
    body = ''.join('| ' + ' | '.join(str(r.get(c, '')) for c in cols) + ' |\n' for r in rows)
    return header + sep + body

write_csv(OUTPUT_DIR / 'round1c_process_summary.csv', process_rows)
write_csv(OUTPUT_DIR / 'round1c_diagnostic_summary.csv', diagnostic_rows)
write_csv(OUTPUT_DIR / 'round1c_audit_summary.csv', audit_rows)
write_csv(OUTPUT_DIR / 'round1c_allowed_aggregate_overview.csv', overview_rows)

summary_lines = [
    '# Round 1c Allowed Aggregate Summary',
    '',
    'Date: 2026-04-11',
    '',
    '这份摘要只按 `allowed aggregate views` 组织 `Round 1b` 结果，不再对 6 个 smoke cases 直接求混合平均。',
    '',
    '输入文件：',
    '',
    '- [round1c_role_aware_smoke_table.csv](../05_round1b_prep/round1c_role_aware_smoke_table.csv)',
    '- [round1c_role_aware_smoke_subset.csv](../05_round1b_prep/round1c_role_aware_smoke_subset.csv)',
    '- [round1c_aggregate_rules.md](../05_round1b_prep/round1c_aggregate_rules.md)',
    '',
    '## 1. Allowed Aggregate Overview',
    '',
    md_table(overview_rows, ['subset_bucket', 'task_count', 'run_count', 'memory_runs', 'memory_verbalized_runs', 'explicit_use_runs', 'explicit_reject_runs', 'outcome_changed_runs', 'improved_runs', 'degraded_runs']).rstrip(),
    '',
    '## 2. Process Summary',
    '',
    md_table(process_rows, ['task_id', 'target_cluster', 'subset_priority', 'total_runs', 'memory_runs', 'memory_verbalized_runs', 'explicit_use_runs', 'explicit_reject_runs', 'outcome_changed_runs', 'notes']).rstrip(),
    '',
    '## 3. Diagnostic Summary',
    '',
    md_table(diagnostic_rows, ['task_id', 'subset_bucket', 'total_runs', 'outcome_changed_runs', 'improved_runs', 'degraded_runs', 'answer_changed_runs', 'notes']).rstrip(),
    '',
    '## 4. Audit / Boundary Summary',
    '',
    md_table(audit_rows, ['task_id', 'total_runs', 'memory_verbalized_runs', 'report_as', 'notes']).rstrip(),
    '',
    '## 5. Reporting Rule',
    '',
    '- 只允许分别报告 `process summary`、`diagnostic summary`、`audit summary`。',
    '- 不允许再对这 6 个 case 直接报告统一 `EM / F1` 或统一 `relevant vs irrelevant` 平均。',
    '- `wiki_dev_0092` 与 `wiki_dev_6083` 只作 boundary / audit 说明，不进入 transfer evidence。',
]
(OUTPUT_DIR / 'round1c_allowed_aggregate_summary.md').write_text('\n'.join(summary_lines) + '\n')

print('Wrote:', OUTPUT_DIR / 'round1c_process_summary.csv')
print('Wrote:', OUTPUT_DIR / 'round1c_diagnostic_summary.csv')
print('Wrote:', OUTPUT_DIR / 'round1c_audit_summary.csv')
print('Wrote:', OUTPUT_DIR / 'round1c_allowed_aggregate_overview.csv')
print('Wrote:', OUTPUT_DIR / 'round1c_allowed_aggregate_summary.md')


## 5. 预览 summary


In [ ]:
print((OUTPUT_DIR / 'round1c_allowed_aggregate_summary.md').read_text())
